# 01: Data Cleaning

Goal: load all 9 raw Olist CSVs, fix data types (mainly dates), translate product
categories, deduplicate reviews, and pre-aggregate the item-level and payment-level
tables to order-level. Output goes to `data/interim/`.

In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw/"

orders = pd.read_csv(RAW + "olist_orders_dataset.csv")
items = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")
products = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers = pd.read_csv(RAW + "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(RAW + "product_category_name_translation.csv")

print("Loaded 9 tables.")
print("orders:", orders.shape, "| items:", items.shape, "| payments:", payments.shape)
print("reviews:", reviews.shape, "| customers:", customers.shape, "| products:", products.shape)
print("sellers:", sellers.shape, "| geo:", geo.shape, "| cat_translation:", cat_translation.shape)

Loaded 9 tables.
orders: (99441, 8) | items: (112650, 7) | payments: (103886, 5)
reviews: (99224, 7) | customers: (99441, 5) | products: (32951, 9)
sellers: (3095, 4) | geo: (1000163, 5) | cat_translation: (71, 2)


## Sanity check: order status and missing dates

Before touching anything, confirm the two things that will break downstream
analysis if ignored: not every order is `delivered`, and delivery dates have gaps.

In [2]:
print(orders["order_status"].value_counts())
print("\nMissing values per column:\n", orders.isna().sum())


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing values per column:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


## Parse dates

All timestamp columns are read in as strings by default. Convert them to real
datetimes so we can subtract them later (e.g. delivery delay = delivered date −
estimated date). `errors="coerce"` turns unparseable values into `NaT` instead of
crashing.

In [3]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Translate product categories (with fallback)

There are 73 unique category names in `products` but only 71 rows in the
translation table, so 2 categories have no English match. A plain left-join
would leave those as `NaN`. Instead, fall back to the original Portuguese name
so we never lose rows, and fill any remaining nulls (products with no category
at all — 610 of them) as `"unknown"`.

In [4]:
products = products.merge(cat_translation, on="product_category_name", how="left")
products["product_category_name_english"] = (
    products["product_category_name_english"].fillna(products["product_category_name"])
)
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

print("Categories with no translation (now falling back to Portuguese):")
print(products.loc[
    products["product_category_name_english"] == products["product_category_name"],
    "product_category_name"
].unique())

Categories with no translation (now falling back to Portuguese):
['cool_stuff' 'pet_shop' 'consoles_games' 'market_place' 'la_cuisine'
 'audio' 'dvds_blu_ray' 'pc_gamer'
 'portateis_cozinha_e_preparadores_de_alimentos']


## Collapse geolocation to one row per zip prefix

The raw geolocation table has ~1M rows because each zip prefix has many lat/lng
points (different streets/neighborhoods). Joining this directly onto orders would
multiply row counts massively. Average lat/lng per zip prefix first.

In [5]:
geo_agg = (
    geo.groupby("geolocation_zip_code_prefix")
       .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
       .reset_index()
)
print(f"Reduced geolocation from {len(geo):,} rows to {len(geo_agg):,} rows (1 per zip prefix).")

Reduced geolocation from 1,000,163 rows to 19,015 rows (1 per zip prefix).


## Deduplicate reviews

A very small number of orders have more than one review row. Keep only the
earliest one per order so `order_id` is unique in the reviews table (this is
what lets us do a clean 1:1 merge later).

In [6]:
before = len(reviews)
reviews = reviews.sort_values("review_creation_date").drop_duplicates(subset="order_id", keep="first")
print(f"Reviews: {before:,} -> {len(reviews):,} after dropping duplicate order_id rows.")

Reviews: 99,224 -> 98,673 after dropping duplicate order_id rows.


## Aggregate payments and items to order-level

Both `order_payments` and `order_items` have multiple rows per order (multiple
payment methods, multiple line items). Since our analysis is at the order level,
collapse both to one row per `order_id` now, rather than repeatedly grouping
inside every later notebook.

In [7]:
payments_agg = (
    payments.groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        n_payment_methods=("payment_type", "nunique"),
        main_payment_type=("payment_value", lambda x: payments.loc[x.idxmax(), "payment_type"])
    )
    .reset_index()
)

items_agg = (
    items.groupby("order_id")
    .agg(
        n_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        n_sellers=("seller_id", "nunique"),
        n_products=("product_id", "nunique")
    )
    .reset_index()
)

print("payments_agg:", payments_agg.shape)
print("items_agg:", items_agg.shape)

payments_agg: (99440, 4)
items_agg: (98666, 6)


## Save cleaned/interim tables

In [8]:
orders.to_csv("../data/interim/orders_clean.csv", index=False)
products.to_csv("../data/interim/products_clean.csv", index=False)
geo_agg.to_csv("../data/interim/geo_agg.csv", index=False)
reviews.to_csv("../data/interim/reviews_clean.csv", index=False)
items_agg.to_csv("../data/interim/items_agg.csv", index=False)
payments_agg.to_csv("../data/interim/payments_agg.csv", index=False)

print("Saved 6 interim tables to data/interim/")

Saved 6 interim tables to data/interim/
